# 2. Faster R-CNN - Two-Stage Object Detection

## Lich su phat trien (Two-Stage Detectors)

### R-CNN (2014) - Ý tuong goc
- **Pipeline**: Selective Search (2000 region proposals) -> CNN feature extraction (AlexNet) -> SVM classifier + Bbox regressor
- **Van de**: Rat cham (~47s/image) vi phai chay CNN 2000 lan cho moi image
- **Dong gop**: Chung minh CNN co the dung cho object detection

### Fast R-CNN (2015) - Cai tien toc do
- **Cai tien chinh**: Chay CNN 1 lan tren toan bo image -> RoI Pooling de extract features cho moi region
- **Dong gop**: RoI Pooling layer, Multi-task loss (classification + bbox regression cung luc)
- **Van de con lai**: Selective Search van la bottleneck (2s/image)

### Faster R-CNN (2015) - End-to-end
- **Dot pha**: Thay Selective Search bang **Region Proposal Network (RPN)** - mot mang neural nho
- **Kien truc**: Backbone (VGG/ResNet) -> RPN -> RoI Pooling -> Classification + Regression
- **Toc do**: ~5 FPS (nhanh hon 10x so voi R-CNN)
- **Day la model chung ta se cai dat**

## Kien truc Faster R-CNN

```
Input Image
    |
[Backbone CNN] -----> Feature Map (shared)
    |                       |
    |               [RPN - Region Proposal Network]
    |                       |
    |               Region Proposals (top-N)
    |                       |
    +-----> [RoI Pooling] <-+
                |
        [FC Layers]
           /        \
    [Classifier]  [Bbox Regressor]
    (N+1 classes)  (4 * N offsets)
```

In [ ]:
import sys
sys.path.append("../src")

import os
import time
import json
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from dataset import (
    VOC_CLASSES, IDX_TO_CLASS, NUM_CLASSES,
    get_voc_datasets, collate_fn
)
from evaluate import (
    evaluate_at_multiple_ious, measure_inference_time,
    build_confusion_matrix, print_results, save_results
)
from visualize import draw_boxes, plot_per_class_ap, plot_confusion_matrix

%matplotlib inline

# Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

DATA_ROOT = "../data"
RESULTS_DIR = "../results/faster_rcnn"
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2.1 Tao Model

Su dung Faster R-CNN voi backbone ResNet-50 + FPN (Feature Pyramid Network) tu torchvision.
- **Pretrained**: Load weights da train tren COCO, sau do fine-tune tren VOC
- **Thay doi head**: Thay classifier head tu 91 classes (COCO) sang 21 classes (VOC + background)

In [ ]:
def create_faster_rcnn(num_classes=NUM_CLASSES, pretrained_backbone=True):
    """
    Tao Faster R-CNN model voi ResNet-50 + FPN backbone.
    
    Kien truc chi tiet:
    - Backbone: ResNet-50 + Feature Pyramid Network (FPN)
      -> Trich xuat features o nhieu scale (P2-P5)
    - RPN: Region Proposal Network
      -> Sinh ~2000 region proposals tu anchor boxes
      -> Anchor sizes: [32, 64, 128, 256, 512]
      -> Anchor ratios: [0.5, 1.0, 2.0]
    - RoI Align: Trich xuat fixed-size features (7x7) cho moi proposal
    - Head: 2 FC layers -> classifier + bbox regressor
    """
    # Load pretrained model (COCO weights)
    model = fasterrcnn_resnet50_fpn(pretrained=pretrained_backbone)
    
    # Thay classifier head cho VOC (21 classes)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

model = create_faster_rcnn()
model.to(DEVICE)

# In thong tin model
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel structure:")
print(f"  Backbone: ResNet-50 + FPN")
print(f"  RPN anchor sizes: {model.rpn.anchor_generator.sizes}")
print(f"  RPN anchor ratios: {model.rpn.anchor_generator.aspect_ratios}")
print(f"  RoI output size: {model.roi_heads.box_roi_pool.output_size}")
print(f"  Num classes: {NUM_CLASSES} (20 VOC + 1 background)")

## 2.2 Load Data va Training

### Hyperparameters:
- **Optimizer**: SGD (momentum=0.9, weight_decay=0.0005) - theo paper goc
- **Learning Rate**: 0.005 (thap hon paper vi fine-tune, khong train tu dau)
- **LR Scheduler**: StepLR (giam 10x sau moi 3 epochs)
- **Batch size**: 4 (gioi han boi GPU memory)
- **Epochs**: 10 (du de fine-tune tu pretrained weights)

In [ ]:
# Load datasets
train_dataset, val_dataset = get_voc_datasets(DATA_ROOT, "2012")

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True,
    num_workers=2, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False,
    num_workers=2, collate_fn=collate_fn
)

print(f"Train: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"Val:   {len(val_dataset)} images ({len(val_loader)} batches)")

In [ ]:
def train_one_epoch(model, data_loader, optimizer, device, epoch):
    """Train 1 epoch, tra ve average loss."""
    model.train()
    total_loss = 0
    num_batches = 0

    progress = tqdm(data_loader, desc=f"Epoch {epoch+1}")
    for images, targets in progress:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass - Faster R-CNN tu tinh loss khi train mode
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        num_batches += 1

        progress.set_postfix({
            "loss": f"{losses.item():.4f}",
            "loss_cls": f"{loss_dict['loss_classifier'].item():.4f}",
            "loss_box": f"{loss_dict['loss_box_reg'].item():.4f}",
            "loss_rpn_cls": f"{loss_dict['loss_objectness'].item():.4f}",
            "loss_rpn_box": f"{loss_dict['loss_rpn_box_reg'].item():.4f}",
        })

    return total_loss / num_batches


@torch.no_grad()
def get_predictions(model, data_loader, device, score_threshold=0.05):
    """Chay inference tren toan bo dataset, tra ve predictions va ground truths."""
    model.eval()
    all_predictions = []
    all_ground_truths = []

    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for output, target in zip(outputs, targets):
            # Filter theo score threshold
            mask = output["scores"] >= score_threshold
            pred = {
                "boxes": output["boxes"][mask].cpu(),
                "labels": output["labels"][mask].cpu(),
                "scores": output["scores"][mask].cpu(),
            }
            gt = {k: v.cpu() if hasattr(v, "cpu") else v for k, v in target.items()}

            all_predictions.append(pred)
            all_ground_truths.append(gt)

    return all_predictions, all_ground_truths

In [ ]:
# ---- Training ----
NUM_EPOCHS = 10
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

# Optimizer - SGD theo paper goc cua Faster R-CNN
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

# LR scheduler - giam 10x moi 3 epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Training loop
train_losses = []
val_maps = []

for epoch in range(NUM_EPOCHS):
    # Train
    avg_loss = train_one_epoch(model, train_loader, optimizer, DEVICE, epoch)
    train_losses.append(avg_loss)
    lr_scheduler.step()
    
    # Evaluate moi 2 epochs hoac epoch cuoi
    if (epoch + 1) % 2 == 0 or epoch == NUM_EPOCHS - 1:
        predictions, ground_truths = get_predictions(model, val_loader, DEVICE)
        results = evaluate_at_multiple_ious(predictions, ground_truths, NUM_CLASSES)
        val_maps.append(results["mAP_50"])
        print(f"\n  Epoch {epoch+1}: loss={avg_loss:.4f}, mAP@0.5={results['mAP_50']:.4f}, mAP@0.5:0.95={results['mAP_50_95']:.4f}")
    else:
        print(f"\n  Epoch {epoch+1}: loss={avg_loss:.4f}")
    
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")

# Save model
model_path = os.path.join(RESULTS_DIR, "faster_rcnn_voc.pth")
torch.save(model.state_dict(), model_path)
print(f"\nModel saved to {model_path}")

## 2.3 Training Loss Curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# Loss curve
color1 = "#2196F3"
ax1.plot(range(1, len(train_losses) + 1), train_losses, "o-", color=color1, label="Train Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

# mAP curve (on second y-axis)
if val_maps:
    ax2 = ax1.twinx()
    color2 = "#4CAF50"
    eval_epochs = [i for i in range(1, NUM_EPOCHS + 1) if i % 2 == 0 or i == NUM_EPOCHS]
    ax2.plot(eval_epochs[:len(val_maps)], val_maps, "s-", color=color2, label="Val mAP@0.5")
    ax2.set_ylabel("mAP@0.5", color=color2)
    ax2.tick_params(axis="y", labelcolor=color2)
    ax2.set_ylim(0, 1.0)

plt.title("Faster R-CNN Training Progress")
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "training_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2.4 Danh gia chi tiet tren Validation Set

In [ ]:
# Final evaluation
predictions, ground_truths = get_predictions(model, val_loader, DEVICE)
final_results = evaluate_at_multiple_ious(predictions, ground_truths, NUM_CLASSES)

# In ket qua
print_results(final_results, "Faster R-CNN (ResNet-50 + FPN)")

# Save results
save_results(final_results, os.path.join(RESULTS_DIR, "eval_results.json"))

In [ ]:
# Per-class AP chart
fig = plot_per_class_ap(final_results, "Faster R-CNN")
plt.savefig(os.path.join(RESULTS_DIR, "per_class_ap.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Confusion Matrix
conf_matrix = build_confusion_matrix(predictions, ground_truths, iou_threshold=0.5, num_classes=NUM_CLASSES)
fig = plot_confusion_matrix(conf_matrix, title="Faster R-CNN - Confusion Matrix")
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2.5 Do Inference Speed

In [ ]:
# Do inference speed
speed_results = measure_inference_time(model, val_loader, DEVICE, num_warmup=5, num_runs=50)
print(f"Faster R-CNN Inference Speed:")
print(f"  Average time: {speed_results['avg_time_ms']:.1f} ms/batch")
print(f"  FPS: {speed_results['fps']:.1f}")

# Luu speed vao results
final_results["fps"] = speed_results["fps"]
final_results["avg_inference_ms"] = speed_results["avg_time_ms"]
final_results["total_params"] = sum(p.numel() for p in model.parameters())
save_results(final_results, os.path.join(RESULTS_DIR, "eval_results.json"))

## 2.6 Visualization - Detection Results

In [ ]:
# Hien thi ket qua detection tren mot so anh val
np.random.seed(123)
sample_indices = np.random.choice(len(val_dataset), 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
for ax, idx in zip(axes.flat, sample_indices):
    image, target = val_dataset[idx]
    
    # Run inference
    model.eval()
    with torch.no_grad():
        output = model([image.to(DEVICE)])[0]
    
    # Ve len anh
    img_np = image.permute(1, 2, 0).numpy()
    ax.imshow(img_np)
    
    import matplotlib.patches as patches
    
    # Ground truth (do, net dut)
    for box, label in zip(target["boxes"], target["labels"]):
        x1, y1, x2, y2 = box.numpy()
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                  edgecolor="red", facecolor="none", linestyle="--")
        ax.add_patch(rect)
    
    # Predictions (xanh, score >= 0.5)
    mask = output["scores"].cpu() >= 0.5
    for box, label, score in zip(output["boxes"].cpu()[mask], output["labels"].cpu()[mask], output["scores"].cpu()[mask]):
        x1, y1, x2, y2 = box.numpy()
        cls_name = IDX_TO_CLASS[label.item()]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                  edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1-3, f"{cls_name}:{score:.2f}", fontsize=7, color="white",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="green", alpha=0.8))
    
    n_pred = mask.sum().item()
    n_gt = len(target["labels"])
    ax.set_title(f"GT={n_gt} (red) | Pred={n_pred} (green)", fontsize=10)
    ax.axis("off")

plt.suptitle("Faster R-CNN Detection Results (Red=GT, Green=Pred)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "detection_samples.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2.7 Phan tich Faster R-CNN

### Diem manh:
- **Do chinh xac cao**: Two-stage approach cho phep tinh chỉnh bbox chinh xac hon
- **Tot voi small objects**: RPN + RoI Align giu duoc chi tiet spatial
- **FPN**: Multi-scale features giup detect objects o nhieu kich thuoc

### Diem yeu:
- **Cham**: Two-stage pipeline (RPN -> RoI -> Head) ton nhieu thoi gian
- **Phuc tap**: Nhieu hyperparameters (anchor sizes, NMS thresholds, RPN/RoI thresholds)
- **Khong phu hop real-time**: FPS thap, khong dung duoc cho video real-time

### So sanh voi paper goc:
- Paper goc (VGG-16 backbone): mAP@0.5 = 73.2% tren VOC 2007
- Implementation nay (ResNet-50 + FPN): ket qua se duoc so sanh o notebook benchmark

### Cai tien tu Fast R-CNN -> Faster R-CNN:
1. **RPN thay Selective Search**: End-to-end training, nhanh hon 10x
2. **Shared features**: Backbone features duoc dung chung cho RPN va detection head
3. **Anchor mechanism**: Thay vi sliding window, dung anchor boxes o nhieu scales/ratios